# 3. The Deformable Mirror and the Closed AO Loop

Part 3 of the tutorial series (`01_Dataset.ipynb`, `02_WFSAndPreprocessing.ipynb`). This notebook introduces the last physical component of the pipeline, `AI4AO.DeformableMirror`, and the closed-loop feedback mechanic itself: measure the residual wavefront, apply a correction with some gain/leak, and repeat.

There is deliberately no neural network yet. To isolate the loop mechanic from the question of *how* you measure the residual phase, the "reconstruction" step here is an oracle: it projects the true residual phase directly onto the DM's actuator basis (`z_inv`, defined below), as if you had perfect knowledge of the wavefront. That's obviously not available on a real bench — the WFS frame is still propagated and displayed at every step, but only for visualization, not used to compute the correction. Replacing this oracle with a real reconstructor that decodes the WFS frame is exactly what notebook 5 does.

## Configuration and the familiar pieces

Same `wfs_params_exp.py` config, `PhaseDataset`, and `PyramidWFS` as before. We turn on `generateClosedLoop` immediately, since the whole point of this notebook is closed-loop dynamics. The WFS is frozen (`.eval()`) since nothing in this notebook is being calibrated or trained — we're only running it forward.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import torch

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, imshow_multiple

device = 'cuda'  # set to "cpu" if CUDA is not available

paramfile = 'wfs_params_exp.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

wfs = PyramidWFS(WFSParams, device)
wfs.eval()


framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

## The deformable mirror

`DeformableMirror` converts a vector of per-actuator commands into a physical phase surface, via Gaussian/Moffat-shaped influence functions (`dm.IF`) on an actuator grid built from `DMParams['Nactuator']`. Change the value of `dm.offset_to_fit_number_of_actuators` to get the correct number of actuators in the DM. Its constructor already lays out the actuator grid and influence functions (`MakeActGrid`/`MakeZonalModes`); we only need to re-center them on the pupil afterwards — masking by the pupil and subtracting each actuator's mean over it — so DM commands don't inject piston from outside the aperture. As with the WFS, we freeze it (`.eval()`) since we're not calibrating anything here — freezing is also what allows the in-place re-centering below.

### The `DMParams` fields

- `Nactuator`: actuators across the DM's nominal square grid diameter (already used in notebook 1 to set the fitting-PSD cutoff). Together with `offset_to_fit_number_of_actuators` (a constructor argument, not a `DMParams` key — see below) it decides which grid points fall inside the DM's circular aperture, hence the actual actuator count.
- `Nmodes`: number of command coefficients the DM basis exposes (the `coefs` dimension in `dm(coefs)`); here it's expected to equal the actuator count for a purely zonal basis (`M2C = eye(Nmodes)` below).
- `moffatParam`: shape parameter of each actuator's Moffat-profile influence function (`dm.IF`) — controls how quickly the influence falls off away from the actuator center.
- `signedAmplitude`: scales the influence-function amplitude/sign; reparameterized internally (`sign`, at `1e-6` scale) so it sits at a similar magnitude to the DM's other learnable parameters.
- `MechCoupling`: mechanical coupling between adjacent actuators (fraction of one actuator's stroke seen at its neighbor), used to set the influence function's width.
- `FlipLeftRight`/`FlipTopBottom`: whether the actuator grid is mirrored along each axis, to match a real DM's wiring/orientation.

In [ ]:
dm = DeformableMirror(WFSParams, DMParams, device)
dm.offset_to_fit_number_of_actuators = 0.1
dm.eval()

## Zernike Modes

You can obtain a modes-to-command (M2C) matrix that projects the first nModes Zernike modes into the DM influence functions. You can get it as `M2C = dm.MakeZernikeM2C()`. You can pass the argument nModes, which defaults to the one given in the DMdict.

In [ ]:
nModes = 64
M2C = dm.MakeZernikeM2C(nModes=nModes)
modes = dm(M2C.T)

IF_grid = torch.zeros((dm.Nact**2, dm.Nres, dm.Nres), device=device, dtype=torch.float32)
IF_grid[dm.grid.flatten()] = dm.IF

imshow_multiple([
    {"tensor": IF_grid, "title": "Influence functions grid"},
    {"tensor": modes, "title": "Grid of Zernike modes"}
])
plt.show()

### Effect of `offset_to_fit_number_of_actuators`

`dm.grid` (set by `MakeActGrid`) keeps the actuator-grid points inside a circle of radius `Nact/2 + offset_to_fit_number_of_actuators`. `Nactuator` alone only fixes the pitch of the square grid the circle is cut out of; the offset is what decides exactly how many of those grid points end up counted as real actuators (`dm.totalAct`) — a small change shifts which corner/edge points fall just inside or outside the circular aperture. Below, `0` uses the grid's exact nominal radius, while `-0.1`/`0.1` shrink/enlarge it slightly.

In [ ]:
offsets = [0, -0.2, 0.2]

fig, axes = plt.subplots(1, len(offsets), figsize=(12, 4))
for ax, offset in zip(axes, offsets):
    dm.offset_to_fit_number_of_actuators = offset  # triggers MakeActGrid()
    ax.imshow(dm.grid.cpu())
    ax.set_title(f"offset = {offset}\n{dm.totalAct.item()} actuators")
    ax.axis('off')
plt.show()

dm.offset_to_fit_number_of_actuators = 0.1  # restore the value used by the rest of this notebook

## An oracle reconstructor: `M2C` and `z_inv`

`M2C` converts a vector of actuator coefficients into DM commands — here it's just the identity, since this synthetic instrument has no calibrated modal basis (a KL `M2C_KL.npy` file, as the bench-calibrated instruments have). `z_inv`, the pseudo-inverse of `dm(M2C.T)`, does the reverse: it projects a *known*, full-resolution phase screen onto actuator coefficients. Given the true residual phase, `residual_phase.flatten(...) @ z_inv` is exactly the command that would null it out — the oracle reconstruction used below.

In [ ]:
M2C = torch.eye(DMParams["Nmodes"], device=device)
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

## Closing the loop

At each step:
1. `residual_phase = phaseGT - phase_reconstructed` — what's left after the *previous* correction.
2. The WFS propagates `residual_phase` to a detector frame — shown purely for intuition; the oracle below ignores it.
3. `Ze = residual_phase.flatten(...) @ z_inv` — the oracle's estimate of the correction needed, computed from the true residual phase.
4. `z_estimated = z_estimated * leak + gain * z_buffer` — the usual leaky-integrator update: `gain` controls how much of the latest estimate is added in each step, `leak` lets old commands decay (a leak below 1 slowly bleeds off DM position, trading a little steady-state error for robustness against runaway drift).
5. `phase_reconstructed = dm(z_estimated @ M2C.T)` — the new correction actually applied to the mirror, which becomes the baseline for the *next* step's residual.

`loop_gain`/`loop_leak` come from the batch itself (drawn per-sample by `PhaseDataset`), so this rollout uses whatever loop tuning was drawn for it.

In [ ]:
n_steps = 60

batch = dataset[0]
phaseGT = batch["phase"]
gain, leak = batch["loop_gain"], batch["loop_leak"]
wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

z_estimated = torch.zeros(phaseGT.shape[0], DMParams["Nmodes"], device=device)
z_buffer = torch.zeros_like(z_estimated)
phase_reconstructed = torch.zeros_like(phaseGT)

phases, residuals, reconstructed, wfs_frames, psfs = [], [], [], [], []

with torch.no_grad():
    for i in range(n_steps):
        if i > 0:
            phaseGT = dataset[i]["phase"]

        residual_phase = phaseGT - phase_reconstructed
        wfs_frame = wfs(residual_phase)
        psf = wfs.GetPSF(residual_phase, sampling=4, fov=20)

        Ze = torch.matmul(residual_phase.flatten(start_dim=-2), z_inv)  # oracle "measurement"
        z_estimated = z_estimated * leak + gain * z_buffer
        z_buffer = torch.clone(Ze)

        phase_reconstructed = dm(z_estimated @ M2C.T)

        phases.append(phaseGT)
        residuals.append(residual_phase)
        reconstructed.append(phase_reconstructed)
        wfs_frames.append(wfs_frame)
        psfs.append(psf)

fig, axes = imshow_multiple(
    [
        {"tensor": phases[0], "title": "Input phase", "same_scale": True},
        {"tensor": reconstructed[0], "title": "Reconstructed phase", "scale_reference": phases[0]},
        {"tensor": residuals[0], "title": "Residual phase", "scale_reference": phases[0]},
        {"tensor": wfs_frames[0], "title": "WFS frame"},
        {"tensor": psfs[0], "title": "PSF", "same_scale": True},
    ],
    max_channel_number=9,
)  # You can change this number to show more or less images


def update(i):
    imshow_multiple(    
        [
            {"tensor": phases[i], "title": "Input phase", "same_scale": True},
            {"tensor": reconstructed[i], "title": "Reconstructed phase", "scale_reference": phases[i]},
            {"tensor": residuals[i], "title": "Residual phase", "scale_reference": phases[i]},
            {"tensor": wfs_frames[i], "title": "WFS frame"},
            {"tensor": psfs[i], "title": "PSF", "same_scale": True},
        ],
            fig=fig, 
            axes=axes, 
            max_channel_number=9
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

anim = FuncAnimation(fig, update, frames=n_steps, interval=80, blit=True)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())